<a href="https://colab.research.google.com/github/Hrishikesh6666/Agentic-AI/blob/main/Swiggy_Annual_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install all required packages
!pip install -q langchain langchain-community langchain-openai
!pip install -q faiss-cpu
!pip install -q pypdf pdfplumber
!pip install -q sentence-transformers
!pip install -q gradio
!pip install -q openai tiktoken
!pip install -q groq langchain-groq  # Alternative free LLM

print('✅ All packages installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 13.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.3/331.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.7 MB/s eta 0:00:00
✅ All packages installed!


In [2]:
!pip install -U langchain-text-splitters
!pip install -U langchain-core langchain-community langchain-openai langchain-text-splitters faiss-cpu
!pip install langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.13
    Uninstalling langchain-core-1.2.13:
      Successfully uninstalled langchain-core-1.2.13


In [3]:
from dotenv import load_dotenv
import os
from openai import OpenAI

In [6]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set.")

MODEL = 'gpt-4o-mini'
openai = OpenAI(api_key=api_key)

In [7]:
# Upload your PDF file
from google.colab import files
import os

print("📂 Please upload the Swiggy Annual Report PDF...")
uploaded = files.upload()

# Get the uploaded filename
pdf_filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {pdf_filename}")
print(f"📊 File size: {os.path.getsize(pdf_filename) / (1024*1024):.2f} MB")

📂 Please upload the Swiggy Annual Report PDF...


Saving Annual-Report-FY-2023-24 (1).pdf to Annual-Report-FY-2023-24 (1).pdf

✅ Uploaded: Annual-Report-FY-2023-24 (1).pdf
📊 File size: 12.85 MB


In [8]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_path, mode="page")
documents = loader.load()

# Add 1-based page indexing
for doc in documents:
    doc.metadata["page"] = doc.metadata.get("page", 0) + 1

print("Total pages:", len(documents))

Total pages: 170


In [9]:
import pdfplumber
import re
from langchain_core.documents import Document

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    print("langchain.text_splitter not found. Attempting to install langchain...")
    !pip install -q langchain
    from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document

def load_and_clean_pdf(filepath):
    """Load PDF and extract clean text with page metadata."""
    documents = []

    with pdfplumber.open(filepath) as pdf:
        total_pages = len(pdf.pages)
        print(f"📖 Total pages: {total_pages}")

        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if not text or len(text.strip()) < 50:
                continue  # Skip blank/image-only pages

            # Clean the text
            text = clean_text(text)

            if text.strip():
                doc = Document(
                    page_content=text,
                    metadata={
                        "source": filepath,
                        "page": i + 1,
                        "total_pages": total_pages
                    }
                )
                documents.append(doc)

    return documents


def clean_text(text):
    """Clean extracted text."""

    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)

    text = re.sub(r'^\d{1,3}$', '', text, flags=re.MULTILINE)

    text = re.sub(r'Annual Report \d{4}-\d{2,4}', '', text, flags=re.IGNORECASE)
    text = text.strip()
    return text


# Load the PDF
print("⏳ Loading and processing PDF...")
raw_documents = load_and_clean_pdf(pdf_filename)
print(f"✅ Loaded {len(raw_documents)} pages with text content")

⏳ Loading and processing PDF...
📖 Total pages: 170
✅ Loaded 30 pages with text content


In [11]:
# Split documents into chunks
print("✂️ Splitting documents into chunks...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(raw_documents)


print(f"✅ Created {len(chunks)} text chunks")
print(f"📊 Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")

# Preview a sample chunk
print("\n--- Sample Chunk (Page 1) ---")
print(chunks[0].page_content[:500])
print(f"\nMetadata: {chunks[0].metadata}")

✂️ Splitting documents into chunks...
✅ Created 70 text chunks
📊 Average chunk size: 800 characters

--- Sample Chunk (Page 1) ---
INDEX
S. NO. Document Name Page No.
1. Corporate Information 1-2
Board’s Report
2. 3-29
Independent Auditor’s Report on
3. Standalone Financial Statement 30-42
4. Standalone Financial Statement 43-98
Independent Auditor’s Report on
5. 99-108
Consolidated Financial Statement
Consolidated Financial Statement
6. 109-167

Metadata: {'source': 'Annual-Report-FY-2023-24 (1).pdf', 'page': 2, 'total_pages': 170}


In [15]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
vector_store = FAISS.from_documents(chunks, embeddings)

print("Vector store built.")

Vector store built.


In [16]:
import os

# ==========================================
# CHECK API KEY
# ==========================================
if "OPENAI_API_KEY" not in os.environ:
    raise ValueError("Set OPENAI_API_KEY in environment.")

# ==========================================
# IMPORTS (NO langchain.chains)
# ==========================================
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# ==========================================
# LLM
# ==========================================
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"],
)

embeddings = OpenAIEmbeddings(
    api_key=os.environ["OPENAI_API_KEY"]
)

print("LLM initialized")

# ==========================================
# SPLIT
# ==========================================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)
print("Chunks:", len(chunks))

# ==========================================
# VECTOR STORE
# ==========================================
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Vectorstore ready")

# ==========================================
# STRICT PROMPT
# ==========================================
prompt = ChatPromptTemplate.from_template("""
You are an expert analyst for Swiggy's Annual Report.

Rules:
1. Answer strictly using the provided context.
2. Do not use external knowledge.
3. If not found, say:
   "I could not find this information in the Swiggy Annual Report."
4. Do not hallucinate.
5. Cite page numbers if present.

Context:
{context}

Question:
{question}

Answer:
""")

# ==========================================
# FORMAT DOCUMENTS
# ==========================================
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ==========================================
# LCEL RAG PIPELINE (NO CHAINS MODULE)
# ==========================================
rag_pipeline = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG pipeline ready")

# ==========================================
# ASK QUESTION
# ==========================================
response = rag_pipeline.invoke(
    "What was Swiggy's total revenue in FY23?"
)

print("\nAnswer:\n", response)

LLM initialized
Chunks: 77
Vectorstore ready
RAG pipeline ready

Answer:
 Swiggy's total revenue in FY23 was ₹53,613 million (page 3).


In [17]:
def ask_question(query):
    print("\n" + "="*60)
    print(f"Question: {query}")
    print("="*60)

    answer = rag_pipeline.invoke(query)

    print("\nAnswer:\n")
    print(answer)

    return answer


test_questions = [
    "What is Swiggy's total revenue for FY23?",
    "Who are the key management personnel of Swiggy?",
    "What are Swiggy's main business segments?",
]

for q in test_questions:
    ask_question(q)


Question: What is Swiggy's total revenue for FY23?

Answer:

Swiggy's total revenue for FY23 is ₹53,613 million. (Page 3)

Question: Who are the key management personnel of Swiggy?

Answer:

The key management personnel of Swiggy are:

1. Sriharsha Majety - Managing Director & Group CEO (DIN: 06680073)
2. Lakshmi Nandan Reddy Obul - Whole time Director – Head of Innovation (DIN: 06686145) 

This information is found on the first page of the provided context.

Question: What are Swiggy's main business segments?

Answer:

I could not find this information in the Swiggy Annual Report.


In [18]:
import gradio as gr

# ==========================================
# BACKEND FUNCTION
# ==========================================
def gradio_query(question, show_context, history):
    if not question.strip():
        return history, ""

    try:
        # 1️⃣ Retrieve documents manually
        docs = retriever.invoke(question)

        # 2️⃣ Generate answer (LCEL expects string input)
        answer = rag_pipeline.invoke(question)

        # 3️⃣ Format context
        context_text = ""
        if show_context and docs:
            context_parts = []
            for i, doc in enumerate(docs, 1):
                page = doc.metadata.get("page", "N/A")
                snippet = doc.page_content[:500].replace("\n", " ")
                context_parts.append(
                    f"**[{i}] Page {page}:**\n> {snippet}..."
                )
            context_text = "\n\n".join(context_parts)

        # 4️⃣ Update chat history
        history = history or []
        history.append((question, answer))

        return history, context_text

    except Exception as e:
        error_msg = f"⚠️ Error: {str(e)}"
        history = history or []
        history.append((question, error_msg))
        return history, ""


def clear_chat():
    return [], "", ""


SAMPLE_QUESTIONS = [
    "What is Swiggy's total revenue for FY23?",
    "What is Swiggy's net loss?",
    "Who are the board of directors?",
    "What are Swiggy's key business segments?",
    "What are the major risks mentioned?",
]

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="orange"),
) as demo:

    gr.Markdown(
        """
        # 🍊 Swiggy Annual Report — RAG Q&A
        Ask questions grounded strictly in the Annual Report.
        """
    )

    with gr.Row():

        # LEFT: CHAT
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=450)

            question_input = gr.Textbox(
                placeholder="Ask a question...",
                lines=2
            )

            submit_btn = gr.Button("Ask 🔍", variant="primary")
            clear_btn = gr.Button("Clear 🗑️")

            show_context_toggle = gr.Checkbox(
                label="Show Retrieved Context",
                value=True
            )

        # RIGHT: CONTEXT
        with gr.Column(scale=2):
            context_box = gr.Markdown(
                value="Retrieved context will appear here."
            )

    # SAMPLE BUTTONS
    gr.Markdown("### 💡 Sample Questions")

    with gr.Row():
        for q in SAMPLE_QUESTIONS:
            btn = gr.Button(q)
            btn.click(fn=lambda x=q: x, outputs=question_input)


    submit_btn.click(
        fn=gradio_query,
        inputs=[question_input, show_context_toggle, chatbot],
        outputs=[chatbot, context_box],
    ).then(lambda: "", outputs=question_input)

    question_input.submit(
        fn=gradio_query,
        inputs=[question_input, show_context_toggle, chatbot],
        outputs=[chatbot, context_box],
    ).then(lambda: "", outputs=question_input)

    clear_btn.click(
        fn=clear_chat,
        outputs=[chatbot, context_box, question_input]
    )


print("🚀 Launching Gradio app...")
demo.launch(share=True)

/tmp/ipython-input-556/1401501690.py:61: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipython-input-556/1401501690.py:76: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=450)
/tmp/ipython-input-556/1401501690.py:76: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=450)


🚀 Launching Gradio app...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://78415980a29ee12883.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
